## 01: Skrip Ingesti Data (Data Ingestion) - Versi Cloud

Tujuan notebook ini adalah untuk memuat "Knowledge Base" kita ke **ChromaDB Cloud**.

**Proses:**
1.  **Muat Kredensial**: Baca `.env` dari folder `backend/` untuk mendapatkan API Key, Tenant, dan Database.
2.  **Siapkan Data**: Kita akan menggunakan data teks sederhana (dummy).
3.  **Siapkan Embeddings**: Kita akan memuat model embedding lokal (`all-MiniLM-L6-v2`).
4.  **Koneksi & Ingest**: Kita akan terhubung ke CloudClient dan menggunakan LangChain untuk memuat dokumen.

In [22]:
import os
import chromadb
from dotenv import load_dotenv
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# --- TAMBAHAN YANG DIPERLUKAN ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import glob
# --- ---------------------- ---

### Langkah 1: Muat Kredensial & Konfigurasi

In [23]:
# --- KODE BARU (YANG BENAR) ---

# Path ke file .env.
# Logika ini berarti: "Dari folder 'notebooks/' saat ini,
# naik satu level ('..') ke folder root 'RAG',
# lalu masuk ke folder 'backend', dan temukan file '.env'".
dotenv_path = os.path.join('..', 'backend', '.env')

# Periksa apakah file .env ada sebelum mencoba memuatnya
if not os.path.exists(dotenv_path):
    raise FileNotFoundError(f"File .env tidak ditemukan di path yang diharapkan: {os.path.abspath(dotenv_path)}")

# Muat file .env
load_dotenv(dotenv_path=dotenv_path)

# Baca variabel yang kita butuhkan
CHROMA_API_KEY = os.getenv('CHROMA_API_KEY')
CHROMA_TENANT = os.getenv('CHROMA_TENANT')
CHROMA_DATABASE = os.getenv('CHROMA_DATABASE')
EMBEDDING_MODEL_NAME = os.getenv('EMBEDDING_MODEL', 'all-MiniLM-L6-v2')

COLLECTION_NAME = "chart_knowledge" # Nama 'folder' di dalam database Anda

if not CHROMA_API_KEY:
    raise ValueError("CHROMA_API_KEY ditemukan sebagai None. Pastikan nilainya ada di dalam file .env.")

print("Variabel .env berhasil dimuat.")
print(f"Tenant: {CHROMA_TENANT}, Database: {CHROMA_DATABASE}")

Variabel .env berhasil dimuat.
Tenant: 6d4f29ca-1eef-4130-bfb1-1af19306b943, Database: RAG_rope


### Langkah 2 & 3: Siapkan Data & Model Embedding

In [24]:
# --- LANGKAH 2: Muat dan Proses PDF ---
print("Memuat dokumen PDF dari folder /data...")

# Path relatif dari /notebooks ke /data
data_path = "../data/" 
pdf_files = glob.glob(data_path + "*.pdf")

print(f"Menemukan {len(pdf_files)} file PDF.")

all_pages = []
for pdf_file in pdf_files:
    print(f"Memuat: {pdf_file}")
    loader = PyPDFLoader(pdf_file)
    pages = loader.load() # Memuat semua halaman
    all_pages.extend(pages)

print(f"Total halaman PDF yang dimuat: {len(all_pages)}")

# --- LANGKAH 3: Split Dokumen ---
# Kita perlu membagi PDF menjadi potongan kecil (chunks)
# agar efektif untuk RAG.

print("Memulai proses splitting dokumen...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10000,  # <-- NAIKKAN ANGKA INI
    chunk_overlap=700   # <-- Sesuaikan juga ini
)

split_chunks = text_splitter.split_documents(all_pages)

print(f"Dokumen telah di-split menjadi {len(split_chunks)} chunks.")

# --- LANGKAH 4: Siapkan Model Embedding ---
# Kode ini sama seperti sebelumnya (jangan dihapus)
print("Memuat model embedding... (Mungkin butuh waktu saat pertama kali)")
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)
print("Model embedding berhasil dimuat.")

Memuat dokumen PDF dari folder /data...
Menemukan 3 file PDF.
Memuat: ../data\practical-fibonacci-methods-for-forex-trading.pdf


Ignoring wrong pointing object 2 65536 (offset 0)
Ignoring wrong pointing object 43 65536 (offset 0)
Ignoring wrong pointing object 56 65536 (offset 0)
Ignoring wrong pointing object 59 65536 (offset 0)
Ignoring wrong pointing object 62 65536 (offset 0)
Ignoring wrong pointing object 97 65536 (offset 0)
Ignoring wrong pointing object 128 65536 (offset 0)
Ignoring wrong pointing object 139 65536 (offset 0)
Ignoring wrong pointing object 168 65536 (offset 0)
Ignoring wrong pointing object 190 65536 (offset 0)
Ignoring wrong pointing object 202 65536 (offset 0)
Ignoring wrong pointing object 231 65536 (offset 0)
Ignoring wrong pointing object 273 65536 (offset 0)
Ignoring wrong pointing object 308 65536 (offset 0)
Ignoring wrong pointing object 319 65536 (offset 0)
Ignoring wrong pointing object 325 65536 (offset 0)
Ignoring wrong pointing object 330 65536 (offset 0)
Ignoring wrong pointing object 335 65536 (offset 0)
Ignoring wrong pointing object 341 65536 (offset 0)
Ignoring wrong poin

Memuat: ../data\technical-analysis-course-cambridge.pdf
Memuat: ../data\TTR.pdf
Total halaman PDF yang dimuat: 280
Memulai proses splitting dokumen...
Dokumen telah di-split menjadi 283 chunks.
Memuat model embedding... (Mungkin butuh waktu saat pertama kali)
Model embedding berhasil dimuat.


### Langkah 4: Koneksi & Ingest ke ChromaDB Cloud

Alih-alih `persist_directory`, kita sekarang akan menggunakan `CloudClient` dan LangChain adapter.

In [25]:
# --- SEL OPSIONAL: HAPUS COLLECTION LAMA ---
print(f"Mencoba menghapus collection lama: {COLLECTION_NAME}...")
try:
    client.delete_collection(name=COLLECTION_NAME)
    print(f"Berhasil menghapus collection: {COLLECTION_NAME}")
except Exception as e:
    print(f"Gagal menghapus atau collection tidak ada: {e}")

Mencoba menghapus collection lama: chart_knowledge...
Berhasil menghapus collection: chart_knowledge


In [26]:
print("Menghubungkan ke ChromaDB Cloud...")

# 1. Inisialisasi Klien Cloud
client = chromadb.CloudClient(
    api_key=CHROMA_API_KEY,
    tenant=CHROMA_TENANT,
    database=CHROMA_DATABASE
)

print(f"Terhubung ke Database: {CHROMA_DATABASE}")

# 2. Gunakan LangChain adapter untuk memuat dokumen
# ... (kode koneksi) ...
print(f"Menyimpan dokumen ke collection: {COLLECTION_NAME}...")
vectorstore = Chroma.from_documents(
    documents=split_chunks, # <-- UBAH INI
    embedding=embeddings,
    client=client, 
    collection_name=COLLECTION_NAME
)

print("--- SELESEI! ---")
print(f"Berhasil menyimpan {len(split_chunks)} chunks ke ChromaDB Cloud.") # <-- UBAH INI

Menghubungkan ke ChromaDB Cloud...
Terhubung ke Database: RAG_rope
Menyimpan dokumen ke collection: chart_knowledge...
--- SELESEI! ---
Berhasil menyimpan 283 chunks ke ChromaDB Cloud.


### Verifikasi (Opsional)

Mari kita coba lakukan pencarian sederhana.

In [27]:
print("Melakukan tes pencarian...")

# 'vectorstore' yang kita buat di atas sudah siap untuk di-query
query = "Apa itu pola bendera?"
results = vectorstore.similarity_search(query, k=1)

if results:
    print(f"Query: {query}")
    print(f"Hasil teratas: {results[0].page_content}")
    print(f"Sumber: {results[0].metadata.get('source')}")
else:
    print("Tes pencarian gagal.")

Melakukan tes pencarian...
Query: Apa itu pola bendera?
Hasil teratas: 42 SMA
Value
A object of the same class as x or price or a vector (if try.xts fails) containing the columns:
SMA Simple moving average.
EMA Exponential moving average.
WMA Weighted moving average.
DEMA Double-exponential moving average.
EVWMA Elastic, volume-weighted moving average.
ZLEMA Zero lag exponential moving average.
VWMA V olume-weighed moving average (same asVWAP).
VWAP V olume-weighed average price (same asVWMA).
VWA Variable-length moving average.
HMA Hull moving average.
ALMA Arnaud Legoux moving average.
Warning
Some indicators (e.g. EMA, DEMA, EVWMA, etc.) are calculated using the indicators’ own
previous values, and are therefore unstable in the short-term. As the indicator receives more data,
its output becomes more stable. See example below.
Note
For EMA, wilder=FALSE (the default) uses an exponential smoothing ratio of2/(n+1), while wilder=TRUE
uses Welles Wilder’s exponential smoothing ratio of 1